In [1]:
import cv2
import numpy as np
import pywt
import torch
from PIL import Image
from torchvision import transforms, datasets

class Preprocessing_ours2:
    def __init__(self, size=32, crop=0, use_channels=['LL', 'LH', 'HL']):
        self.size = size
        self.crop = crop
        self.use_channels = use_channels

    def __call__(self, img):
        # Step 1: Convert PIL image to NumPy array & crop
        img_np = np.array(img.convert("RGB"))
        if self.crop > 0:
            h, w, _ = img_np.shape
            img_np = img_np[self.crop:h - self.crop, self.crop:w - self.crop, :]

        # Step 2: Convert to grayscale
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        # Step 3: 
        coeffs2 = pywt.dwt2(gray, 'haar')
        LL, (LH, HL, HH) = coeffs2

        # Step 4: Resize subbands
        bands = {
            'LL': cv2.resize(LL, (self.size, self.size), interpolation=cv2.INTER_CUBIC),
            'LH': cv2.resize(LH, (self.size, self.size), interpolation=cv2.INTER_CUBIC),
            'HL': cv2.resize(HL, (self.size, self.size), interpolation=cv2.INTER_CUBIC),
            'HH': cv2.resize(HH, (self.size, self.size), interpolation=cv2.INTER_CUBIC),
        }

        # Step 5: Binarize each selected band using Otsu
        tensors = []
        for key in self.use_channels:
            band = bands[key]
            band = cv2.normalize(band, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            _, binary = cv2.threshold(band, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            tensor = transforms.functional.to_tensor(Image.fromarray(binary))  # (1, H, W)
            tensors.append(tensor)

        # Step 6: Stack into multi-channel tensor
        stacked = torch.cat(tensors, dim=0)  # (C, H, W)
        return stacked

# Example: Use LL, LH, HL (3 channels)
image_transform = transforms.Compose([
    Preprocessing_ours2(size=32, crop=2, use_channels=['LL', 'LH', 'HL','HH'])
])

trainset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Train', transform=image_transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=40, shuffle=True, num_workers=2, drop_last=True)

testset  = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Test',  transform=image_transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=40, shuffle=False, num_workers=2, drop_last=True)

In [2]:
print(f"# of trainset = {len(trainset)}")
print(f"# of trainloader = {len(trainloader)}")
target, label = trainset[5000]
print(f"input size: {target.shape}, {label}")     

# of trainset = 8000
# of trainloader = 200
input size: torch.Size([4, 32, 32]), 2


In [3]:
import os
import numpy as np 
import torch.optim as optim   
from tqdm import tqdm 

def train(model, device, trainloader, optimizer, criterion, num_epochs, save_path='./prob2_3_ours2_weight2'):
    os.makedirs(save_path, exist_ok=True)  
    history = []

    for epoch in tqdm(range(num_epochs)):
        model.train()
        epoch_loss, correct, total = 0, 0, 0

        for X, y in trainloader: 
            X = X.to(device);y = y.to(device)

            optimizer.zero_grad()
            predict = model(X)
            loss = criterion(predict, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pred_class = predict.argmax(dim=1)
            correct += (pred_class == y).sum().item()
            total += y.size(0)

        avg_loss = epoch_loss / len(trainloader)
        avg_accuracy = correct / total
        history.append((epoch + 1, avg_loss, avg_accuracy))
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.6f}, Accuracy: {avg_accuracy:.4f}")

        if (epoch + 1) % 10 == 0:
            filename = os.path.join(save_path, f"weight{epoch+1}.pth")
            torch.save(model.state_dict(), filename)
            print(f"Saved checkpoint: {filename}")

    return np.array(history)
    
def test(model, device, test_loader, criterion):
    test_loss = []
    test_accuracy = []
    model.eval()

    with torch.no_grad():
        for X, y in tqdm(test_loader): 
            X = X.to(device)
            y = y.to(device)

            predict = model(X)
            loss = criterion(predict, y)

            pred_class = predict.argmax(dim=1)
            accuracy = (pred_class == y).float().mean()

            test_accuracy.append(accuracy.item())
            test_loss.append(loss.item())

    avg_loss = sum(test_loss) / len(test_loss)
    avg_accuracy = sum(test_accuracy) / len(test_accuracy)
    print(f'test loss : {avg_loss:.4f} / test_accuracy : {avg_accuracy:.4f}')

In [4]:
## resnet model ##
import torchvision.models as models
import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ours2 = models.resnet18()
num_ftrs = ours2.fc.in_features
ours2.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
ours2.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4),
)
ours2 = ours2.to(device)

In [6]:
# 하이퍼파라미터 설정
num_epochs = 50
lr = 0.001
optimizer = torch.optim.Adam(ours2.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()

# 학습 실행
history = train(ours2, device, trainloader, optimizer, criterion, num_epochs)

  2%|▉                                           | 1/50 [00:13<11:14, 13.77s/it]

Epoch [1/50] - Loss: 1.340211, Accuracy: 0.4070


  4%|█▊                                          | 2/50 [00:28<11:21, 14.20s/it]

Epoch [2/50] - Loss: 1.114578, Accuracy: 0.5069


  6%|██▋                                         | 3/50 [00:42<11:02, 14.09s/it]

...

Epoch [48/50] - Loss: 0.032630, Accuracy: 0.9889


 98%|██████████████████████████████████████████▏| 49/50 [11:16<00:13, 13.57s/it]

Epoch [49/50] - Loss: 0.047446, Accuracy: 0.9846


100%|███████████████████████████████████████████| 50/50 [11:30<00:00, 13.80s/it]

Epoch [50/50] - Loss: 0.029608, Accuracy: 0.9904
Saved checkpoint: ./prob2_3_ours2_weight2/weight50.pth


In [8]:
import torch
import torch.nn as nn
import torchvision.models as models
import os
model_paths = [
    './prob2_3_ours2_weight2/weight10.pth',
    './prob2_3_ours2_weight2/weight20.pth',
    './prob2_3_ours2_weight2/weight30.pth',
    './prob2_3_ours2_weight2/weight40.pth',
    './prob2_3_ours2_weight2/weight50.pth',
]
criterion = torch.nn.CrossEntropyLoss()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

for path in model_paths:
    model = models.resnet18()
    model.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 4)
    )
    model.load_state_dict(torch.load(path, map_location=device))
    model = model.to(device)
    test(model, device, testloader, criterion)

100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 13.79it/s]


test loss : 1.4558 / test_accuracy : 0.5750


100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 13.98it/s]


test loss : 2.1095 / test_accuracy : 0.5408


100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 13.84it/s]


test loss : 2.4769 / test_accuracy : 0.5579


100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 13.62it/s]


test loss : 2.4169 / test_accuracy : 0.5974


100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 14.30it/s]

test loss : 2.5025 / test_accuracy : 0.5882
